# Building A Chatbot

In this Video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for :

- Conversation RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions



In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv() # loading all the env vars

groq_api_key = os.getenv("GROQ_API_KEY")
print(groq_api_key)

In [ ]:
from langchain_groq import ChatGroq

model=ChatGroq(model="moonshotai/kimi-k2-instruct-0905" , groq_api_key=groq_api_key)
model

In [ ]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Krish and I am a Chief AI Engineer")])

In [ ]:
from langchain_core.messages import AIMessage

model.invoke(

    [
        HumanMessage(content="Hi, My name is Krish and I am a Chief AI Engineer"),
        AIMessage(content='<think>\nOkay, let\'s talk about what I should do next. The user introduced themselves as Krish, a Chief AI Engineer. That\'s pretty cool! They might be looking for a conversation that\'s both technical and engaging.\n\nFirst, I need to acknowledge their introduction. A simple "Hi Krish!" would be friendly. Then, I should ask them about their work to show interest. Maybe something like, "It\'s great to meet you! Could you tell me more about your role as a Chief AI Engineer?" That opens the door for them to share their experiences.\n\nI should also consider the possible directions the conversation could take. Krish might want to discuss technical challenges they\'re facing, or they might be interested in exploring new ideas in AI. I should be prepared to ask follow-up questions based on their initial response. For example, if they mention working on machine learning models, I could ask about the specific applications they\'re developing or the tools they\'re using.\n\nLet me check if there are any other ways to make this conversation engaging. I should avoid making assumptions about their specific field within AI engineering. They could be working on anything from computer vision to natural language processing. Keeping the questions open-ended allows them to guide the discussion.\n\nI should also consider if they\'re looking for collaboration opportunities or just want to discuss general AI concepts. I\'ll keep my response flexible enough to accommodate either possibility. The key is to create an environment where they feel comfortable sharing their expertise and experiences.\n\nHmm, maybe I should also think about the tone. Since they\'re in a leadership position, I want to maintain a professional yet approachable tone. Avoiding overly casual language while still being friendly seems like the right balance.\n\nWait, should I mention something about my capabilities as an AI assistant? Maybe not at first. Let them set the agenda. If they ask about my capabilities later, I can elaborate.\n\nLet me review my response once more to ensure it\'s welcoming and encourages further discussion. Yes, it seems good. I\'ll go with that initial friendly greeting and an open question about their work to start the conversation.\n</think>\n\nHi Krish! It\'s great to meet you! Could you tell me more about your role as a Chief AI Engineer? I\'d love to hear about the projects you\'re working on or any challenges you\'re tackling in the field.'),
        HumanMessage(content="Hey, whats my name and what do I do ?")
    ]
)

## Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model,get_session_history)

In [ ]:
config = {"configurable":{"session_id":"chat1"}}

In [ ]:
response = with_message_history.invoke(
   [
       HumanMessage(content="Hi, My name is Krish and I am a Chief AI Engineer")
   ],
   config=config
)

In [ ]:
response.content

In [ ]:
with_message_history.invoke(
   [
       HumanMessage(content="what's my name ?")
   ],
   config=config
)

In [ ]:
## Change the config --> changing the session id
config2 = {"configurable":{"session_id":"chat2"}}

response=with_message_history.invoke(
   [
       HumanMessage(content="what's my name ?")
   ],
   config=config2
)

print(response.content)

# Prompt Templates

Prompt Templates help to turn raw user information into a format that LLM can work with. in this case, the raw user input is just a message, which we are passing to the LLM. Let''s now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages

In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the questions to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model

In [ ]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")]})

In [ ]:
with_message_history= RunnableWithMessageHistory(chain,get_session_history)



In [ ]:
config3 = {"configurable":{"session_id":"chat3"}}

response = with_message_history.invoke(
    [HumanMessage(content="Hi My name is Krish")],
    config3
)

response

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the questions to the best of your ability in {language}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model

In [ ]:
response = chain.invoke({"messages":[HumanMessage(content="Hi my name is Krish!")],"language":"Hindi"})
response.content

In [ ]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [ ]:
config4 = {"configurable":{"session_id":"chat4"}}

response = with_message_history.invoke(
    {"messages":[HumanMessage(content="Hi, I am ABhishek")],"language":"Hindi"}
    ,
    config=config4
)

response.content

In [ ]:


response = with_message_history.invoke(
    {"messages":[HumanMessage(content="Hi, what is my name")],"language":"Hindi"}
    ,
    config=config4
)

response.content

# Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system messages and whether to allow partial messages.

In [ ]:
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    BaseMessage,
    SystemMessage,
    trim_messages,
)

messages = [
    SystemMessage("you're a good assistant, you always respond with a joke."),
    HumanMessage("i wonder why it's called langchain"),
    AIMessage(
        'Well, I guess they thought "WordRope" and "SentenceString" just '
        "didn't have the same ring to it!"
    ),
    HumanMessage("and who is harrison chasing anyways"),
    AIMessage(
        "Hmmm let me think.\n\nWhy, he's probably chasing after the last "
        "cup of coffee in the office!"
    ),
    HumanMessage("what do you call a speechless parrot"),
]

trimmer=trim_messages(
   
    max_tokens=70,
    strategy="last",
    token_counter=model,
    # Most chat models expect that chat history starts with either:
    # (1) a HumanMessage or
    # (2) a SystemMessage followed by a HumanMessage
    start_on="human",
    # Usually, we want to keep the SystemMessage
    # if it's present in the original history.
    # The SystemMessage has special instructions for the model.
    include_system=True,
    allow_partial=False,
)

trimmer.invoke(messages)

In [ ]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | model
)

response = chain.invoke({"messages":messages + [HumanMessage(content="what was my first question ?")]})

response.content

In [ ]:
# Lets wrap this in the message history

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config = {"configurable":{"session_id":"chat6"}}

In [ ]:
response = with_message_history.invoke(
    {
       "messages":[HumanMessage(content="what was my first question ?")]
    },
    config = config
)

response.content